<p style="text-align:right">Фазлыев А.А, МОиАИС 20.01-1</p>
<h1 style="text-align:center">Лабораторная работа №6</h1>
<h2 style="text-align:center">Тема: Сплайн интерполяция</h2>

### 1. Постановка задачи

#### Определение сплайна
Сплайном $S_m(x)$ называется определенная на  $[a,b]$ функция, принадлежащая классу $C^l[a,b]$, $l$ - раз непрерывно дифференцируемых функций, такая, что на каждом промежутке $[x_{k-1},x_k],  k=1,\dots,n$  это многочлен $m$-ой степени.

#### Посроение кубичческого сплайна
Пусть на $[a, b]$ задана непрерывная функция $f(x)$. Введем сетку $a = x_0 < \dots < x_n = b$ и обозначим $f_i = f(x_i), i = 0, \dots, n$

Сплайном для функции $f(x)$ и узлов $\{x_i\}_{i=0}^n$ называется функция $S(x)$, удовлетворяющая следующим условиям:
1. На каждом сегменте $[x_{i-1}, x_i], i=1, \dots, n$ функция $S(x)$ является многочленом третьей степени $S(x) = \{S_i(x)\}, i = 1, \dots, n$
2. Функция S(x), а так же ее первая $S'(x)$ и вторая $S''(x)$ производные непрерывны на $[a, b]$
3. $S(x_i) = f(x_i), i = 0, \dots, n$ - условие интерполирования

В промежутке между парой соседних узлов интерполяционная функция является многочленом 3-й степени, который удобно записать в виде:
$S_i(x) = a_i + b_i(x - x_{i - 1}) + c_i(x - x_{i - 1})^2 + d_i(x - x_{i - 1})^3$

Используя условия кубического сплайна, а так же предположение о нулевой кривизне графика на концах, можно прийти к системе уравнений

\begin{cases}
c_0 = 0 \\
hc_{i - 1} + 4hc_i + hc_{i + 1} = 3(\frac{f_i - f_{i-1}}{h} - \frac{f_{i - 1} - f_{i-2}}{h}), 2 \leq i \leq n \\
c_n = 0
\end{cases}

Матрица этой системы - трехдиагональная, поэтому может быть эффективно решена методом прогонки

#### Метод прогонки для трехдиагональных матриц
Модификация метода Гаусса, использующуюся для решения СЛУ вида $Ax = b$ где A - матрица вида
\begin{pmatrix}
a_1 & b_1 \\
c_1 & a_2 & b_2 \\
    & c_2 & \dots & \dots \\
    && \dots & \dots & b_{n - 1} \\
    &&       & c_{n - 1} & a_n \\
\end{pmatrix}

Алгоритм заключается в следующем:

1. Вычисляются прогоночные коэффициенты $\alpha_i$ и $\beta_i$ для $i = 1, \dots, n - 1$ (прямой ход)

 $\alpha_i = \frac{c_i}{b_i - a_i\alpha_{i - 1}}$
 $\beta_i = \frac{a_i\beta_{i - 1} - d_i}{b_i - a_i\alpha_{i - 1}}$
 
2. Вычисляются неизвестные $x_i$ для $i = 1 \dots n - 1$ (обратный ход)

 $x_i = \frac{c_i}{b_i - a_i\alpha{i - 1}}$
 $x_{i + 1} = \frac{a_i\beta_{i - 1} - d_i}{b_i - a_i\alpha_{i - 1}}$

#### Порядок выполнения работы
Функция $y = f(x) = e^{-x} - x^3$ задана таблицей своих значений.
Для функции $y = f(x)$ построить интерполяционный кубический сплайн дефекта 1
Результаты представить в виде таблицы значений в узлах сетки $\tilde x_i = x_0 + i\frac{h}{2}, i = 0, \dots, 2n$

|$\tilde x_i = x_0 + i\frac{h}{2}$|$f(\tilde x_i)$|$S(\tilde x_i)$|$\Delta = |f(\tilde x_i) - S(\tilde x_i)|$|
|---------------------------------|---------------|----------------|-------------------------------------------|
|                                 |               |                |                                           |



In [1]:
import numpy as np
from ipywidgets import interact, FloatRangeSlider, BoundedIntText
from matplotlib import pyplot as plt

Метод прогонки для решения СЛУ

In [2]:
def tridiag_solve(a, b, c, d):
    n = len(d)
    alpha = np.zeros(n)
    beta = np.zeros(n)

    alpha[0] = c[0] / b[0]
    beta[0] = d[0] / b[0]
    
    # Прямой ход
    for i in range(1, n):
        alpha[i] = alpha[i] / (b[i] - a[i - 1] * alpha[i - 1])
        beta[i] = (d[i] - a[i - 1] * beta[i - 1]) / (b[i] - a[i - 1] * alpha[i - 1])

    x = np.zeros(n)
    x[-1] = beta[-1]

    # Обратный ход
    for i in range(n - 2, -1, -1):
        x[i] = beta[i] - alpha[i] * x[i + 1]

    return x

Интерполяция с помощью кубического сплайна

In [3]:
def cubic_spline(x, y):
    n = len(x)
    h = np.diff(x)
 
    a = np.zeros(n - 1)
    b = np.zeros(n)
    c = np.zeros(n - 1)
    d = np.zeros(n)

    # Задаем коэффициенты для трехдиагональной матрицы
    a[:-1] = h[:-1] / (h[:-1] + h[1:])
    b[...] = 2
    c[1:] = h[1:] / (h[:-1] + h[1:])
    d[1:-1] = 6 * ((y[2:] - y[1:-1]) / h[1:] - (y[1:-1] - y[:-2]) / h[:-1]) / (h[1:] + h[:-1])
    
    # Используем прогонку для решения трехдиагональной матрицы
    m = tridiag_solve(a, b, c, d)

    # Вычисляем коэффициенты сплайна
    coeff = np.transpose([
        y[:-1],
        (y[1:] - y[:-1] - (m[1:] + 2 * m[:-1]) * h * h / 6),
        m[:-1] * h * h / 2,
        (m[1:] - m[:-1]) * h * h / 6,
    ])
    
    def spline(x_):
        # Находим индекс интервала, в котором находится x_
        i = np.searchsorted(x, x_) - 1
        i = np.clip(i, 0, n - 2) # Ограничиваем i, чтобы не выйти за границы массива
        
        # Вычисляем значение сплайна для x_
        z = (x_ - x[i]) / h[i]
        
        return np.sum(
            coeff[i] * z.reshape(-1, 1) ** np.arange(4),
            axis=1
        )

    return spline


In [4]:
@interact(
    a_b=FloatRangeSlider(
        value=[-5, 5],
        min=-10,
        max=10,
        step=0.1,
        description='$[a, b]$:',
    ),
    n=BoundedIntText(
        min=2,
        step=1,
        description='$n$',
    ),
)
def show_plot(a_b, n):
    # Исходные данные
    f = lambda x: np.exp(-x) - x**3
    a, b = a_b
    x = np.linspace(a, b, n)
    
    # Вычисляем значения функции в узлах интерполяции
    y = f(x)
    
    # Вычисляем сплайн
    spline = cubic_spline(x, y)

    # Вычисляем значения сплайна на сетке
    x_t = np.linspace(a, b, n * 2)
    y_t = f(x_t)
    y_s = spline(x_t)
    
    # Строим график
    x_plot = np.linspace(a, b, n * 10)
    y_plot = f(x_plot)
    
    fig1, ax1 = plt.subplots(figsize=(10, 5))
    ax1.plot(x_plot, y_plot, label="Исходная функция")
    ax1.plot(x_t, y_t, "o", label="Узлы интерполяции")
    ax1.plot(x_t, y_s, label="Сплайн")
    ax1.legend()

    fig2, ax2 = plt.subplots(figsize=(10, 5))
    ax2.table(
        cellText=np.transpose([
            x_t, y_t, y_s, abs(y_t - y_s)
        ]).round(3),
        colLabels=[
            r'$\tilde x_i = x_0 + i\frac{h}{2}$',
            r'$f(\tilde x_i)$',
            r'$S(\tilde x_i)$',
            r'$\Delta = |f(\tilde x_i) - S(\tilde x_i)|$'
        ],
        loc="top"
    )
    ax2.axis("off")

    fig1, fig2

interactive(children=(FloatRangeSlider(value=(-5.0, 5.0), description='$[a, b]$:', max=10.0, min=-10.0), Bound…